# Quantum Medical Image Classification: Exploratory Analysis & Results

This notebook provides interactive exploratory data analysis, quantum circuit inspection, training progression, and comparative evaluation for the **PneumoniaMNIST** benchmark.

In [1]:
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import pennylane as qml

from src.data.dataset import load_pneumoniamnist, get_dataset_statistics
from src.models.quantum_circuit import create_quantum_circuit
from src.models.classical_cnn import ClassicalCNN
from src.models.classical_hybrid import ClassicalHybridCNN
from src.models.quantum_hybrid import HybridQuantumCNN

print("Environment initialized successfully.")

## 1. Dataset Verification and Statistics

We inspect the official PneumoniaMNIST dataset splits (Train, Validation, Test) and class distribution.

In [2]:
train_raw, val_raw, test_raw = load_pneumoniamnist("../data")
stats = get_dataset_statistics(train_raw, val_raw, test_raw)
for split in ["train", "val", "test"]:
    s = stats[split]
    print(f"{split.upper()}: Total={s["total"]}, Normal={s["normal"]} ({s["normal_pct"]:.2f}%), Pneumonia={s["pneumonia"]} ({s["pneumonia_pct"]:.2f}%)")
print(f"Computed training pos_weight (for BCEWithLogitsLoss): {stats["pos_weight"]:.4f}")

## 2. Visualizing Medical Images (Normal vs. Pneumonia)

Displaying pediatric chest radiographs at 28x28 grayscale resolution.

In [3]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
normal_idx = np.where(train_raw.labels.squeeze() == 0)[0][:4]
pneumonia_idx = np.where(train_raw.labels.squeeze() == 1)[0][:4]
for i, idx in enumerate(normal_idx):
    axes[0, i].imshow(train_raw.imgs[idx], cmap="gray")
    axes[0, i].set_title(f"Normal ({idx})")
    axes[0, i].axis("off")
for i, idx in enumerate(pneumonia_idx):
    axes[1, i].imshow(train_raw.imgs[idx], cmap="gray")
    axes[1, i].set_title(f"Pneumonia ({idx})")
    axes[1, i].axis("off")
plt.suptitle("Sample Chest X-rays from PneumoniaMNIST", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Quantum Circuit Architecture and Visualization

Inspection of the 4-qubit variational circuit with AngleEmbedding, RX/RY/RZ rotations, ring entanglement, and Pauli-Z measurements.

In [4]:
dev = qml.device("default.qubit", wires=4)
circuit = create_quantum_circuit(n_qubits=4, n_layers=2, rotation="Y", dev=dev)
dummy_x = np.array([0.5, -0.2, 0.8, -0.4])
dummy_w = np.zeros((2, 4, 3))
print(qml.draw(circuit)(dummy_x, dummy_w))

## 4. Parameter Counts Across Architectures

Comparing parameter efficiency between Classical CNN (baseline), CNN + MLP (classical control), and Hybrid QNN.

In [5]:
m_a = ClassicalCNN()
m_b = ClassicalHybridCNN()
m_c = HybridQuantumCNN()
print(f"Model A (Classical CNN):     {sum(p.numel() for p in m_a.parameters()):,} parameters")
print(f"Model B (CNN + MLP Control):  {sum(p.numel() for p in m_b.parameters()):,} parameters")
print(f"Model C (Hybrid QNN):         {sum(p.numel() for p in m_c.parameters()):,} parameters")

## 5. Experimental Test Set Evaluation Summary

Loading test evaluation metrics computed on the held-out test split (N=624).

In [6]:
with open("../outputs/metrics/test_metrics.json", "r") as f:
    results = json.load(f)
import pandas as pd
summary_rows = []
for model, data in results.items():
    m = data["test_metrics_default_threshold"]
    summary_rows.append({
        "Model": model,
        "Accuracy (%)": f"{m["accuracy"]*100:.2f}%",
        "Precision": f"{m["precision"]:.4f}",
        "Recall": f"{m["recall"]:.4f}",
        "Specificity": f"{m["specificity"]:.4f}",
        "F1-Score": f"{m["f1"]:.4f}",
        "ROC-AUC": f"{m["roc_auc"]:.4f}",
        "Training Time (s)": f"{data["training_time_sec"]:.2f}",
        "Latency (ms/img)": f"{data["latency_ms_per_image"]:.2f}",
    })
df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))